# HyperKT — Stage 9C: Five-seed knowledge-tracing baselines
**Goal:** Train DKT, DKVMN, SAKT and AKT-inspired baseline implementations using the **same Stage 2 train/validation splits**, five seeds, and identical validation metric definitions. **No test-set predictions are performed.**

**Important methodology:** The SAKT/AKT models here are explicitly **reference-inspired compact implementations**, not verified reproductions of official authors’ repositories. Before describing them as published baselines in a journal, compare against the authors’ implementations and hyperparameter protocols. DKT and DKVMN also require independent implementation checks. The Stage 8B HyperKT architecture differs in capacity, so results are not automatically capacity-matched. Sequence-window event uniqueness remains unresolved.

**Runtime:** L4 GPU preferred, T4 GPU supported. 20 model trainings may take several hours. Google Drive persistence supports resumption.

In [ ]:
%pip -q install numpy pandas scikit-learn scipy matplotlib tqdm

In [ ]:
import os,io,zipfile,json,hashlib,random,time,platform,gc
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, f1_score
from scipy import stats
from tqdm.auto import tqdm
print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS=[42,123,2024,3407,7777]; VARIANTS=['DKT','DKVMN','SAKT','AKT_inspired']

## 1. Upload Stage 2 archive and configure Drive output
Upload `hyperkt_stage2(2).zip`. Do not upload or open the test split for evaluation.

In [ ]:
from google.colab import files,drive
from google.colab import files as colab_files
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/HyperKT_Stage9C');ROOT.mkdir(parents=True,exist_ok=True)
uploaded=colab_files.upload()
assert len(uploaded)==1 and next(iter(uploaded)).lower().endswith('.zip'),'Upload the Stage 2 ZIP only'
zip_name,zip_bytes=next(iter(uploaded.items())); z=zipfile.ZipFile(io.BytesIO(zip_bytes))
assert {'train_sequences.npz','val_sequences.npz','manifest.json','skill_vocab.json'}.issubset(z.namelist())
source_hash=hashlib.sha256(zip_bytes).hexdigest()
def read_npz(name):
 with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a:return {k:a[k].copy() for k in a.files}
train=read_npz('train_sequences.npz'); val=read_npz('val_sequences.npz')
# The test_sequences.npz member is deliberately not read.
N_SKILLS=int(max(train['target_skill'].max(),val['target_skill'].max(),train['history_skill'].max(),val['history_skill'].max())+1)
print('Stage 2 SHA256:',source_hash,'N_SKILLS:',N_SKILLS,'train/val:',len(train['mask']),len(val['mask']))
assert set(train['user_id']).isdisjoint(set(val['user_id']))
assert int(train['mask'].sum())==222562 and int(val['mask'].sum())==23858,'Unexpected Stage 2 target counts'
assert train['mask'].shape[1]==val['mask'].shape[1]==200


## 2. Data protocol and metrics
All models consume previous interaction and target skill, and predict the **next response**. Padding is ignored in loss and evaluation. AUC and NLL are computed over all valid target positions, not averaged per window. F1 threshold is fixed at 0.5; ECE uses 15 equal-width bins.

In [ ]:
def seed_all(s):
 random.seed(s);np.random.seed(s);torch.manual_seed(s)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(s)
 torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
class SeqDataset(Dataset):
 def __init__(self,d):self.d=d
 def __len__(self):return len(self.d['mask'])
 def __getitem__(self,i):
  return tuple(torch.as_tensor(self.d[k][i],dtype=(torch.bool if k=='mask' else torch.long)) for k in ['history_skill','history_correct','target_skill','target_correct','mask'])
def eval_metrics(y,p):
 y=np.asarray(y,dtype=np.int64);p=np.clip(np.asarray(p,dtype=np.float64),1e-7,1-1e-7)
 bins=np.minimum(np.digitize(p,np.linspace(0,1,16)[1:-1]),14)
 ece=sum((bins==b).mean()*abs(y[bins==b].mean()-p[bins==b].mean()) for b in range(15) if (bins==b).any())
 return dict(auc=float(roc_auc_score(y,p)),nll=float(log_loss(y,p,labels=[0,1])),brier=float(brier_score_loss(y,p)),f1=float(f1_score(y,p>=.5)),ece15=float(ece),n_targets=len(y))
def causal_mask(t,device):return torch.triu(torch.ones(t,t,dtype=torch.bool,device=device),diagonal=1)


## 3. Baseline models
DKT uses LSTM hidden states and selects the current target skill from skill logits. DKVMN uses static key memory and dynamic value updates, with per-step read-before-write. SAKT uses causal cross-attention from target skill queries to past interaction keys/values. The AKT-inspired model adds exponential positional distance bias and learned interaction embeddings; it **does not claim exact AKT monotonic attention or Rasch parameterization**. All models exclude future response information.

In [ ]:
class DKT(nn.Module):
 def __init__(self,n,d=128,dropout=.1):
  super().__init__();self.n=n;self.emb=nn.Embedding(n*2,d);self.rnn=nn.LSTM(d,d,batch_first=True);self.drop=nn.Dropout(dropout);self.out=nn.Linear(d,n)
 def forward(self,hs,hc,ts):
  x=self.emb(hs+hc*self.n);h,_=self.rnn(x);return self.out(self.drop(h)).gather(-1,ts.unsqueeze(-1)).squeeze(-1)
class DKVMN(nn.Module):
 def __init__(self,n,d=128,m=32,dropout=.1):
  super().__init__();self.n=n;self.d=d;self.m=m
  self.key=nn.Parameter(torch.randn(m,d)*.05);self.init_val=nn.Parameter(torch.randn(m,d)*.05)
  self.q=nn.Embedding(n,d);self.inter=nn.Embedding(n*2,d)
  self.erase=nn.Linear(d,d);self.add=nn.Linear(d,d);self.pred=nn.Sequential(nn.Linear(2*d,d),nn.ReLU(),nn.Dropout(dropout),nn.Linear(d,1))
 def forward(self,hs,hc,ts):
  b,t=hs.shape;mem=self.init_val.unsqueeze(0).expand(b,-1,-1).clone();out=[]
  for j in range(t):
   # Incorporate the observed history interaction at j before predicting target j.
   w=self.inter(hs[:,j]+hc[:,j]*self.n)
   history_attention=torch.softmax(self.q(hs[:,j])@self.key.T,dim=-1)
   erase=torch.sigmoid(self.erase(w));add=torch.tanh(self.add(w))
   mem=mem*(1-history_attention.unsqueeze(-1)*erase.unsqueeze(1))+history_attention.unsqueeze(-1)*add.unsqueeze(1)
   q=self.q(ts[:,j]);a=torch.softmax(q@self.key.T,dim=-1)
   read=(mem*a.unsqueeze(-1)).sum(1)
   out.append(self.pred(torch.cat([q,read],-1)).squeeze(-1))
  return torch.stack(out,dim=1)
class SAKT(nn.Module):
 def __init__(self,n,d=128,heads=8,dropout=.1):
  super().__init__();self.n=n
  self.inter=nn.Embedding(n*2,d);self.skill=nn.Embedding(n,d);self.pos=nn.Embedding(200,d)
  self.attn=nn.MultiheadAttention(d,heads,dropout=dropout,batch_first=True)
  self.norm=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,2*d),nn.ReLU(),nn.Dropout(dropout),nn.Linear(2*d,d));self.norm2=nn.LayerNorm(d);self.out=nn.Linear(d,1)
 def forward(self,hs,hc,ts):
  b,t=hs.shape;pos=torch.arange(t,device=hs.device)
  kv=self.inter(hs+hc*self.n)+self.pos(pos);q=self.skill(ts)+self.pos(pos)
  # history at index j is the interaction immediately preceding target j: diagonal is allowed
  a,_=self.attn(q,kv,kv,attn_mask=causal_mask(t,hs.device),need_weights=False)
  h=self.norm(q+a);h=self.norm2(h+self.ff(h));return self.out(h).squeeze(-1)
class AKTInspired(nn.Module):
 def __init__(self,n,d=128,heads=8,dropout=.1):
  super().__init__();self.n=n;self.heads=heads;self.dk=d//heads
  self.inter=nn.Embedding(n*2,d);self.skill=nn.Embedding(n,d)
  self.q=nn.Linear(d,d);self.k=nn.Linear(d,d);self.v=nn.Linear(d,d)
  self.decay=nn.Parameter(torch.tensor(0.02));self.drop=nn.Dropout(dropout)
  self.norm=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,2*d),nn.ReLU(),nn.Dropout(dropout),nn.Linear(2*d,d));self.norm2=nn.LayerNorm(d);self.out=nn.Linear(d,1)
 def forward(self,hs,hc,ts):
  b,t=hs.shape;query=self.skill(ts);memory=self.inter(hs+hc*self.n)
  def heads(x):return x.view(b,t,self.heads,self.dk).transpose(1,2)
  q=heads(self.q(query));k=heads(self.k(memory));v=heads(self.v(memory))
  logits=q@k.transpose(-1,-2)/(self.dk**.5)
  positions=torch.arange(t,device=hs.device);distance=(positions[:,None]-positions[None,:]).clamp(min=0)
  logits=logits-torch.nn.functional.softplus(self.decay)*distance[None,None,:,:]
  logits=logits.masked_fill(causal_mask(t,hs.device)[None,None,:,:],torch.finfo(logits.dtype).min)
  a=self.drop(torch.softmax(logits,dim=-1));h=(a@v).transpose(1,2).contiguous().view(b,t,-1)
  h=self.norm(query+h);h=self.norm2(h+self.ff(h));return self.out(h).squeeze(-1)
def make_model(name):
 return {'DKT':DKT,'DKVMN':DKVMN,'SAKT':SAKT,'AKT_inspired':AKTInspired}[name](N_SKILLS).to(DEVICE)


## 4. Causality and shape smoke tests
Perturb future observed responses and future target skills. Earlier predictions must not change. Also check finite logits and gradients.

In [ ]:
seed_all(42)
for name in VARIANTS:
 model=make_model(name).eval()
 hs=torch.tensor(val['history_skill'][:2],device=DEVICE,dtype=torch.long);hc=torch.tensor(val['history_correct'][:2],device=DEVICE,dtype=torch.long);ts=torch.tensor(val['target_skill'][:2],device=DEVICE,dtype=torch.long)
 with torch.no_grad():
  original=model(hs,hc,ts);changed_hc=hc.clone();changed_hc[:,100:]=1-changed_hc[:,100:];changed_ts=ts.clone();changed_ts[:,100:]=(changed_ts[:,100:]+3)%N_SKILLS
  modified=model(hs,changed_hc,changed_ts)
 delta=(original[:,:100]-modified[:,:100]).abs().max().item()
 assert original.shape==hs.shape and torch.isfinite(original).all() and delta<1e-5,(name,delta)
 print(name,'causal max delta',delta)
 del model
if torch.cuda.is_available():torch.cuda.empty_cache()


## 5. Training configuration
Use equal training budget (not necessarily equal parameter count). AUC on validation determines best checkpoint. Checkpoints and histories persist in Drive. Run-all resumes completed runs.

In [ ]:
CFG=dict(batch_size=32,lr=2e-4,weight_decay=1e-4,max_epochs=20,patience=5,dropout=.1,hidden=128)
train_ds=SeqDataset(train);val_ds=SeqDataset(val)
val_loader=DataLoader(val_ds,batch_size=CFG['batch_size'],shuffle=False,num_workers=0)
(ROOT/'protocol.json').write_text(json.dumps({'config':CFG,'seeds':SEEDS,'variants':VARIANTS,'stage2_zip_sha256':source_hash,'dataset_train_targets':222562,'dataset_val_targets':23858,'test_evaluated':False,'baseline_status':'independent reference-inspired implementations; not verified official implementations','known_limitation':'cross-window event uniqueness unresolved'},indent=2))
def run_eval(model,loader):
 model.eval();ys=[];ps=[]
 with torch.no_grad():
  for hs,hc,ts,tc,mask in loader:
   hs,hc,ts=hs.to(DEVICE),hc.to(DEVICE),ts.to(DEVICE)
   logits=model(hs,hc,ts);prob=torch.sigmoid(logits)[mask.to(DEVICE)]
   ps.append(prob.float().cpu().numpy());ys.append(tc[mask].numpy())
 return eval_metrics(np.concatenate(ys),np.concatenate(ps))
def train_run(name,seed):
 path=ROOT/f'{name}_seed{seed}';path.mkdir(exist_ok=True)
 if (path/'best.pt').exists() and (path/'best_metrics.json').exists():
  print('Resuming/skipping',name,seed);return json.loads((path/'best_metrics.json').read_text())
 seed_all(seed);model=make_model(name);opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay']);loss_fn=nn.BCEWithLogitsLoss()
 gen=torch.Generator().manual_seed(seed);train_loader=DataLoader(train_ds,batch_size=CFG['batch_size'],shuffle=True,generator=gen,num_workers=0)
 best=-1.;best_metrics=None;bad=0;history=[]
 for epoch in range(1,CFG['max_epochs']+1):
  model.train();losses=[]
  for hs,hc,ts,tc,mask in train_loader:
   hs,hc,ts,tc,mask=[v.to(DEVICE) for v in (hs,hc,ts,tc,mask)]
   opt.zero_grad(set_to_none=True);logits=model(hs,hc,ts)
   loss=loss_fn(logits[mask],tc[mask].float());loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();losses.append(loss.item())
  m=run_eval(model,val_loader);history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),**m});pd.DataFrame(history).to_csv(path/'history.csv',index=False)
  print(f'{name} seed={seed} epoch={epoch} val_auc={m["auc"]:.6f}')
  if m['auc']>best+1e-8:
   best=m['auc'];bad=0;best_metrics={**m,'variant':name,'seed':seed,'best_epoch':epoch,'n_parameters':sum(p.numel() for p in model.parameters())}
   torch.save({'state_dict':model.state_dict(),'variant':name,'seed':seed,'epoch':epoch,'n_skills':N_SKILLS,'config':CFG,'stage2_zip_sha256':source_hash},path/'best.pt')
   (path/'best_metrics.json').write_text(json.dumps(best_metrics,indent=2))
  else:
   bad+=1
   if bad>=CFG['patience']:break
 del model,opt,train_loader;gc.collect()
 if torch.cuda.is_available():torch.cuda.empty_cache()
 return best_metrics


## 6. Run 20 baseline experiments
**Do not change settings after seeing validation performance** without clearly labeling a new experiment.

In [ ]:
results=[]
for seed in SEEDS:
 for name in VARIANTS:
  results.append(train_run(name,seed))
  pd.DataFrame(results).to_csv(ROOT/'all_results.csv',index=False)
assert len(results)==20
print('Completed',len(results),'runs')


## 7. Aggregate validation performance
These are validation results only. Avoid significance claims based solely on multiple correlated seeds on a fixed validation cohort.

In [ ]:
df=pd.read_csv(ROOT/'all_results.csv')
assert len(df)==20 and not df[['variant','seed']].duplicated().any()
summary=df.groupby('variant')[['auc','nll','brier','f1','ece15']].agg(['mean','std'])
summary.to_csv(ROOT/'summary_mean_sd.csv');display(summary)
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4));g=df.groupby('variant')['auc'].agg(['mean','std']).reindex(VARIANTS)
ax.bar(g.index,g['mean'],yerr=g['std'],capsize=4);ax.set_ylabel('Validation ROC-AUC');ax.tick_params(axis='x',rotation=15);fig.tight_layout();fig.savefig(ROOT/'validation_auc.png',dpi=160);plt.show()


## 8. Export results ZIP (checkpoints remain in Drive)
Upload the resulting ZIP here. Keep the Drive checkpoint directory for any later independent verification.

In [ ]:
out=Path('/content/hyperkt_stage9c.zip')
with zipfile.ZipFile(out,'w',compression=zipfile.ZIP_DEFLATED) as outz:
 for file in ROOT.rglob('*'):
  if file.is_file() and file.suffix not in ['.pt','.pth','.ckpt']:
   outz.write(file,arcname=str(file.relative_to(ROOT)))
print('Export:',out,'bytes:',out.stat().st_size)
colab_files.download(str(out))
